# Uber Fare Prediction — Linear Regression & Random Forest

**ML Practical — Notebook**

Run cells from top to bottom.

## 1. Import libraries and load dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.read_csv("uber.csv")
df.head()

## 2. Preprocess data

In [ ]:
# Remove unnecessary column and rows with missing values
df = df.drop(columns=["Unnamed: 0"], errors="ignore").dropna()

# Convert pickup time to datetime and extract useful features
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce", utc=True)
df = df.dropna(subset=["pickup_datetime"])
df["hour"] = df["pickup_datetime"].dt.hour
df["day"] = df["pickup_datetime"].dt.day
df["month"] = df["pickup_datetime"].dt.month
df = df.drop(columns=["pickup_datetime", "key"], errors="ignore")

# Remove invalid fares, coordinates and passenger counts
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 500)]
df = df[df["passenger_count"].between(1, 6)]
df = df[df["pickup_latitude"].between(-90, 90) & df["dropoff_latitude"].between(-90, 90)]
df = df[df["pickup_longitude"].between(-180, 180) & df["dropoff_longitude"].between(-180, 180)]
df.head()

## 3. Detect and remove fare outliers using IQR

In [ ]:
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print("Outlier limits:", lower, upper)
df = df[df["fare_amount"].between(lower, upper)]
print("Rows after outlier removal:", len(df))

## 4. Correlation

In [ ]:
corr = df.corr(numeric_only=True)
display(corr["fare_amount"].sort_values(ascending=False))
plt.figure(figsize=(9,6))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Train Linear Regression and Random Forest

In [ ]:
X = df.drop(columns=["fare_amount"])
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append([name, r2_score(y_test, pred),
                   np.sqrt(mean_squared_error(y_test, pred)),
                   mean_absolute_error(y_test, pred)])
pd.DataFrame(results, columns=["Model", "R2", "RMSE", "MAE"])

## 6. Conclusion

In [ ]:
print("Compare R2, RMSE and MAE. Higher R2 and lower RMSE/MAE indicate better performance.")